In [ ]:
# Fully self-contained training with dynamic wheels-dir discovery + P100 torch install
import os, sys, base64, subprocess, glob, shutil
SCRIPT_B64 = "..."
os.makedirs("/kaggle/working/scripts", exist_ok=True)
with open("/kaggle/working/scripts/train_full_frame_center_detector.py", "wb") as f:
    f.write(base64.b64decode(SCRIPT_B64))

# Find ANY wheels dir available
wheels_dirs = []
for root, dirs, files in os.walk("/kaggle/input"):
    if any(f.endswith(".whl") for f in files):
        wheels_dirs.append(root)
# dedup
wheels_dirs = list(set(wheels_dirs))
print(f"Found wheels dirs: {wheels_dirs}")

def find_wheel_in_any(name_prefix):
    for wd in wheels_dirs:
        for f in os.listdir(wd):
            if f.startswith(name_prefix) and f.endswith(".whl"):
                return wd, f
    return None, None

# --- P100 torch fix ---
import torch
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}, sm_{cap[0]}{cap[1]}, x{torch.cuda.device_count()}")
    if cap[0] < 7:
        # need to reinstall torch that supports sm_60
        wd, fname = find_wheel_in_any("torch-2.5")
        if wd and fname:
            # some wheels have bad version (torch-2.5.1cu118 instead of torch-2.5.1+cu118)
            src = os.path.join(wd, fname)
            if "cu118-cp312" in fname and "+" not in fname:
                tmp = "/tmp/torchfix"; os.makedirs(tmp, exist_ok=True)
                # copy renamed
                fixed = os.path.join(tmp, "torch-2.5.1+cu118-cp312-cp312-linux_x86_64.whl")
                shutil.copy(src, fixed)
                # copy other wheels too
                for other_wd in wheels_dirs:
                    for other_f in os.listdir(other_wd):
                        if other_f.endswith(".whl") and not other_f.startswith("torch-"):
                            shutil.copy(os.path.join(other_wd, other_f), tmp)
                subprocess.check_call([sys.executable, "-m", "pip", "install",
                    "--no-index", "--find-links", tmp, "--force-reinstall", fixed])
            else:
                subprocess.check_call([sys.executable, "-m", "pip", "install",
                    "--no-index", "--find-links", wd, "--force-reinstall", src])
            for k in list(sys.modules): 
                if k.startswith("torch"): del sys.modules[k]
            import torch
            print(f"reinstalled torch: {torch.__version__}")
        else:
            raise RuntimeError("P100 detected but no compatible torch wheel found")

# --- Install zarr + numcodecs ---
try:
    import zarr
except ImportError:
    wd, fname = find_wheel_in_any("zarr")
    assert wd, "No zarr wheel found"
    subprocess.check_call([sys.executable, "-m", "pip", "install",
        "--no-index", "--find-links", wd, "--quiet", "zarr", "numcodecs"])
    import zarr

print(f"torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}, zarr: {zarr.__version__}")

# quick GPU test
if torch.cuda.is_available():
    x = torch.zeros(4, device='cuda') + 1
    print(f"CUDA test OK: {x.sum().item()}")

# Find training data
COMPETITION = "biohub-cell-tracking-during-development"
DATA_DIR = None
for p in [f"/kaggle/input/competitions/{COMPETITION}/train",
          f"/kaggle/input/{COMPETITION}/train"]:
    if os.path.isdir(p): DATA_DIR = p; break
assert DATA_DIR, "training data not found"
print(f"DATA_DIR: {DATA_DIR}, zarrs: {len(glob.glob(f'{DATA_DIR}/*.zarr'))}, geffs: {len(glob.glob(f'{DATA_DIR}/*.geff'))}")


In [ ]:
# Run training
import subprocess, time, sys, os
OUTPUT_DIR = "/kaggle/working/weights/full_frame_center"
os.makedirs(OUTPUT_DIR, exist_ok=True)
cmd = [sys.executable, "/kaggle/working/scripts/train_full_frame_center_detector.py",
       "--data-dir", DATA_DIR, "--output-dir", OUTPUT_DIR,
       "--epochs", "30", "--batch-size", "4", "--seed", "2026",
       "--pool-factor", "4", "--base-channels", "24", "--num-workers", "2"]
print("cmd:", " ".join(cmd)); sys.stdout.flush()
t0 = time.time()
r = subprocess.run(cmd)
print(f"exit: {r.returncode}, elapsed: {(time.time()-t0)/60:.1f} min")
assert r.returncode == 0, "training failed"
for f in sorted(os.listdir(OUTPUT_DIR)):
    p = os.path.join(OUTPUT_DIR, f); print(f"  {f}: {os.path.getsize(p)/1e6:.2f} MB")
